# Lab: Understanding Language Models – From N-grams to Prompt Engineering

## Learning goals
- See how sentence probability connects to next-token prediction.
- Implement an n-gram (bigram) model with smoothing + sampling.
- Use a transformer model for generation and study decoding settings.
- Observe hallucination and mitigate it with better prompts/structure.

## Part 0 — Setup
Run these cells to install packages and import libraries.

In [ ]:
!pip install -q nltk transformers torch

In [ ]:
import math
import re
import random
from collections import Counter, defaultdict

import nltk
from nltk.util import ngrams
from transformers import pipeline

nltk.download("punkt")

## Part 1 — Sentence plausibility and “probability of text”

A language model estimates how likely text is by decomposing it into conditional probabilities (chain rule).

In [ ]:
sentences = [
    "Jane went to the store.",
    "store to Jane went the.",
    "Jane rode an ostrich to the store.",
    "Jane goed to the store.",
    "The store went to Jane.",
    "Jane became the store."
]

for s in sentences:
    print("-", s)

### Tasks
1. **Rank** the sentences from most natural to least natural.
2. Pick **one** sentence that is mainly a **grammar** problem, and **one** sentence that is mainly a **meaning** problem.
3. In 2–3 lines: How could a language model still give *some* probability to weird sentences?

## Part 2 — Build a Bigram Language Model (with smoothing)

We’ll:
- tokenize a tiny corpus
- count bigrams
- compute probabilities with **add-one (Laplace) smoothing**
- sample new text from the bigram model
- compute **perplexity** on test sentences

In [ ]:
text = """
I have a dog whose name is Lucy.
I have two cats.
They like playing with Lucy.
"""

tokens = nltk.word_tokenize(text.lower())
tokens[:30], len(tokens)

In [ ]:
# Build counts
unigram_counts = Counter(tokens)
bigram_counts = Counter(ngrams(tokens, 2))

vocab = sorted(set(tokens))
V = len(vocab)
V, vocab[:10]

### Coding task A — Add-one smoothed bigram probability

Implement:

$$
P(w_2 \mid w_1) = \frac{count(w_1,w_2) + 1}{count(w_1) + V}
$$

In [ ]:
def p_bigram_addone(w1, w2):
    # Your code here
    pass

# Quick sanity check:
print("P(lucy|with) =", p_bigram_addone("with", "lucy"))
print("P(dragon|with) =", p_bigram_addone("with", "dragon"))  # unseen token -> will KeyError unless we handle vocab carefully

### Coding task B — Handle unknown tokens

Our toy vocab is tiny. Let’s add an `<UNK>` token so the model can handle unseen words.

In [ ]:
# Add <UNK> to vocab
UNK = "<UNK>"
if UNK not in vocab:
    vocab.append(UNK)
vocab = sorted(vocab)
V = len(vocab)
V, vocab[:10]

In [ ]:
def normalize_token(w):
    # Your code here
    pass

def p_bigram_addone_safe(w1, w2):
    w1 = normalize_token(w1)
    w2 = w2 if w2 in vocab else UNK
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

print("P(<UNK>|with) =", p_bigram_addone_safe("with", "dragon"))

### Coding task C — Next-word prediction

Write a function that returns the top-k next tokens for a given context word.

In [ ]:
def top_k_next(w1, k=5):
    # Your code here
    pass

top_k_next("with", k=8)

### Coding task D — Sample text from the bigram model

We will sample token by token using the bigram probabilities.

In [ ]:
def sample_next(w1):
    # Your code here
    pass

def generate_bigram(seed="i", max_tokens=25):
    cur = seed.lower()
    out = [cur]
    for _ in range(max_tokens - 1):
        nxt = sample_next(cur)
        out.append(nxt)
        cur = nxt
        if nxt == ".":
            break
    return " ".join(out)

for _ in range(5):
    print(generate_bigram(seed="i"))

### Coding task E — Perplexity

Perplexity measures how “surprised” a model is by a sentence.
Implement perplexity for a tokenized sentence under the bigram model:

$$
PP = \exp\left(-\frac{1}{N}\sum_{i=2}^{N}\log P(w_i \mid w_{i-1})\right)
$$

In [ ]:
def perplexity(tokens_seq):
    # Your code here
    pass

test_1 = nltk.word_tokenize("I have a dog .".lower())
test_2 = nltk.word_tokenize("Lucy have two dog .".lower())

print("PP(test_1) =", perplexity(test_1))
print("PP(test_2) =", perplexity(test_2))

### Questions (2–3 only)
1. Compare `PP(test_1)` vs `PP(test_2)`: which is lower and why?
2. What does add-one smoothing fix? What trade-off does it introduce?
3. When would n-gram models struggle compared to transformers?

## Part 3 — Transformer generation and decoding controls

We’ll use a small transformer (GPT-2) to generate text and study how decoding settings change outputs.

In [ ]:
generator = pipeline("text-generation", model="gpt2")

### Coding task A — Compare decoding strategies

In [ ]:
prompt = "The students opened their"
print("PROMPT:", prompt)

out1 = generator(prompt, max_new_tokens=25, do_sample=False)  # greedy-ish
out2 = # Your code here
out3 = # Your code here

print("\n[No sampling]\n", out1[0]["generated_text"])
print("\n[Sampling t=1.0, top_p=0.9]\n", out2[0]["generated_text"])
print("\n[Sampling t=1.5, top_p=0.95]\n", out3[0]["generated_text"])

### Coding task B — Simple repetition metric

Compute the fraction of repeated 3-grams in the generated output (a crude “repetition” indicator).

In [ ]:
def trigram_repetition_ratio(text):
    # Your code here
    pass

for label, out in [("nosample", out1[0]["generated_text"]), ("t1", out2[0]["generated_text"]), ("t1.5", out3[0]["generated_text"])]:
    print(label, "repetition ratio:", trigram_repetition_ratio(out))

### Questions (2–3 only)
1. Which decoding setting produced the most **creative** output? Which produced the most **stable** output?
2. How did repetition ratio change across settings?
3. Why might sampling help or hurt factuality?

## Part 4 — Hallucination and “self-check” prompting

We’ll provoke a likely hallucination and then ask the model to self-check uncertainty.

In [ ]:
prompt_h = "The inventor of the Vietnamese language was"
raw = generator(prompt_h, max_new_tokens=35, do_sample=True, temperature=1.0, top_p=0.9)[0]["generated_text"]
print(raw)

### Coding task — Ask for uncertainty + evidence

We will ask the model to:
- state uncertainty if needed
- list what it would need to verify

In [ ]:
prompt_check = f"""You are a careful assistant.

Task: Respond to the claim below.
Claim: {prompt_h}

Instructions:
1) If you are not sure the claim is factual, say so clearly.
2) Provide a short answer *only if* you are confident.
3) List 3 things you would verify (sources/keywords) to confirm it.
"""

checked = # Your code here
print(checked)

### Questions (2–3 only)
1. Did the model acknowledge uncertainty? Quote one phrase that signals uncertainty (or note if none).
2. Why do LLMs hallucinate (1–2 sentences)?
3. Give one high-stakes domain where hallucination is especially risky and explain why.

## Part 5 — Prompt engineering with structured outputs

We’ll force the model to return a JSON-like structure and then validate it with Python.

### Coding task A — Create a structured prompt

In [ ]:
def ask_structured(question):
    prompt = f"""Return ONLY valid JSON with keys: "answer", "confidence", "assumptions".
- "confidence" must be one of: "low", "medium", "high"
- If confidence is low, "answer" should be an empty string.
- "assumptions" must be a list of strings.

Question: {question}
"""
    return # Your code here

resp = ask_structured("Summarise recent trends in Vietnamese NLP research.")
print(resp)

### Coding task B — Extract and validate JSON-like output

Because models sometimes include extra text, we’ll extract the first JSON object with a regex and validate basic rules.

In [ ]:
import json

def extract_first_json(text):
    # naive: find first {...} block
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        return None
    return m.group(0)

def validate_payload(payload):
    if not isinstance(payload, dict):
        return False, "Not a dict"
    for k in ["answer", "confidence", "assumptions"]:
        if k not in payload:
            return False, f"Missing key: {k}"
    if payload["confidence"] not in ["low", "medium", "high"]:
        return False, "Bad confidence value"
    if payload["confidence"] == "low" and payload["answer"] != "":
        return False, "Low confidence must have empty answer"
    if not isinstance(payload["assumptions"], list):
        return False, "assumptions must be a list"
    return True, "OK"

jtxt = extract_first_json(resp)
print("EXTRACTED JSON TEXT:\n", jtxt)

if jtxt:
    try:
        obj = json.loads(jtxt)
        ok, msg = validate_payload(obj)
        print("VALID:", ok, msg)
        print("PARSED:", obj)
    except Exception as e:
        print("JSON parse failed:", e)

### Questions (2–3 only)
1. Did the model follow the JSON format? If not, what went wrong?
2. Why does forcing a structure sometimes reduce hallucination?
3. Rewrite this vague prompt to be safer: **"What is the best financial model used today?"**

## Wrap-up
- You built and tested a smoothed bigram LM, sampled from it, and computed perplexity.
- You experimented with transformer decoding controls.
- You saw hallucination and applied uncertainty + structure to mitigate it.